<a href="https://colab.research.google.com/github/Santibareiro27/Inteligencia-Computacional/blob/borges/RA2_LAB2/RA2_Laboratorio_N%C2%B0_2_G8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clasificación de Sentimientos en Reseñas de Google Play
## Comparativa de Modelos: Bag of Words vs. Redes Neuronales Secuenciales

---

**Proyecto:** Automatización del análisis de sentimientos para reseñas de aplicaciones móviles  
**Dominio:** Procesamiento de Lenguaje Natural (NLP) — Clasificación Multiclase  
**Framework:** TensorFlow / Keras + Scikit-Learn  
**Autor:** Senior ML Engineer — Inteligencia Computacional  

---

### Resumen Ejecutivo

Una consultora de análisis de producto enfrenta el problema de escala: la revisión manual de miles de reseñas mensuales de Google Play es inviable operacionalmente. El objetivo de este notebook es diseñar, entrenar y evaluar comparativamente tres arquitecturas de clasificación de texto para mapear cada reseña a una de tres categorías de sentimiento:

| Etiqueta | Rango de Estrellas | Clase Numérica |
|---|---|---|
| **Negativo** | 1 – 2 estrellas | `0` |
| **Neutral**  | 3 estrellas      | `1` |
| **Positivo** | 4 – 5 estrellas  | `2` |

La estrategia experimental sigue el eje **Hipótesis → Experimento → Conclusión**, escalando progresivamente en complejidad: desde un modelo estadístico clásico (TF-IDF + Regresión Logística) hasta arquitecturas neurales profundas que explotan la estructura secuencial del lenguaje (BiLSTM y CNN-1D).

---
## Sección 1 — Configuración del Entorno y Análisis Exploratorio de Datos (EDA)

### 1.1 Importación de Librerías

Se importan las dependencias agrupadas por responsabilidad funcional: manipulación de datos, visualización, preprocesamiento de texto y construcción de modelos. La semilla aleatoria (`SEED = 42`) se fija de forma global para garantizar la reproducibilidad completa de los experimentos.

In [ ]:
# ─── Reproducibilidad ────────────────────────────────────────────────────────
import os, random
import numpy as np
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# ─── Manipulación de datos ────────────────────────────────────────────────────
import pandas as pd
import re
from collections import Counter

# ─── Visualización ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.gridspec import GridSpec
sns.set_theme(style='whitegrid', palette='muted')

# ─── Scikit-Learn ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder

# ─── TensorFlow / Keras ───────────────────────────────────────────────────────
import tensorflow as tf
tf.random.set_seed(SEED)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Embedding, Dropout, Bidirectional, LSTM,
    Conv1D, GlobalMaxPooling1D, Dense, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")
print(f"Pandas version     : {pd.__version__}")
print("✓ Entorno configurado correctamente.")

### 1.2 Carga del Dataset

Se carga el archivo `reviews_limpias.csv`, que contiene dos columnas: `clean_reviews` (texto de la reseña ya normalizado en minúsculas) y `sentiment` (etiqueta categórica: `Negativo`, `Neutral`, `Positivo`). Se realiza una auditoría básica de integridad: detección de valores nulos y duplicados, que de existir podrían sesgar tanto el entrenamiento como la evaluación.

In [ ]:
# ─── Carga ────────────────────────────────────────────────────────────────────
df = pd.read_csv('reviews_limpias.csv')

print("═" * 50)
print(f"  Registros totales : {len(df):,}")
print(f"  Columnas          : {list(df.columns)}")
print("═" * 50)

# ─── Auditoría de calidad ─────────────────────────────────────────────────────
print("\n[Nulos por columna]")
print(df.isnull().sum())

n_duplicados = df.duplicated(subset='clean_reviews').sum()
print(f"\n[Reseñas duplicadas] : {n_duplicados:,}")

# Eliminar nulos y duplicados si existen
df.dropna(subset=['clean_reviews', 'sentiment'], inplace=True)
df.drop_duplicates(subset='clean_reviews', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"\n✓ Dataset final limpio: {len(df):,} registros")
df.head()

### 1.3 Análisis del Desbalance de Clases y Longitud de Reseñas

El Análisis Exploratorio de Datos (EDA) tiene un propósito diagnóstico crítico: **revelar patrones estructurales del corpus que condicionan las decisiones de diseño posteriores**. Se analizan dos dimensiones:

1. **Distribución de clases:** En datasets de reseñas reales de plataformas como Google Play, la distribución suele ser bimodal — con alta concentración en los extremos (1★ y 5★) y una minoría de reseñas neutras (3★). Este desbalance estructural, si no se mitiga explícitamente, provoca que cualquier clasificador tienda a ignorar la clase minoritaria, maximizando accuracy a costa del recall en `Neutral`.

2. **Longitud de las reseñas:** La distribución de longitudes informa directamente el hiperparámetro `MAX_LEN` del padding. Usar un valor demasiado bajo introduce ruido por truncamiento; demasiado alto añade secuencias de ceros que degradan la convergencia del entrenamiento.

In [ ]:
# ─── Mapeo de etiquetas a enteros ─────────────────────────────────────────────
LABEL_MAP = {'Negativo': 0, 'Neutral': 1, 'Positivo': 2}
INV_LABEL = {v: k for k, v in LABEL_MAP.items()}
df['label'] = df['sentiment'].map(LABEL_MAP)

# ─── Longitud en tokens (palabras) ───────────────────────────────────────────
df['n_words'] = df['clean_reviews'].apply(lambda x: len(str(x).split()))

# ─── Figura de EDA ───────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 5))
gs  = GridSpec(1, 3, figure=fig, wspace=0.35)

COLORS = ['#E74C3C', '#F39C12', '#27AE60']
CLASS_NAMES = ['Negativo', 'Neutral', 'Positivo']

# ── Panel 1: Distribución de clases (absoluta) ─────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
counts = df['sentiment'].value_counts().reindex(CLASS_NAMES)
bars = ax1.bar(CLASS_NAMES, counts.values, color=COLORS, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_title('Distribución de Clases', fontsize=12, fontweight='bold')
ax1.set_ylabel('Cantidad de Reseñas')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# ── Panel 2: Distribución de clases (proporcional) ─────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
pcts = (counts / counts.sum() * 100).round(1)
wedges, texts, autotexts = ax2.pie(
    pcts, labels=CLASS_NAMES, colors=COLORS,
    autopct='%1.1f%%', startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for at in autotexts:
    at.set_fontsize(10); at.set_fontweight('bold')
ax2.set_title('Proporción de Clases (%)', fontsize=12, fontweight='bold')

# ── Panel 3: Distribución de longitud por clase ────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
for lbl, color in zip(CLASS_NAMES, COLORS):
    subset = df[df['sentiment'] == lbl]['n_words']
    ax3.hist(subset, bins=50, alpha=0.6, color=color, label=lbl, density=True)

p95 = int(np.percentile(df['n_words'], 95))
ax3.axvline(p95, color='black', linestyle='--', linewidth=1.5, label=f'P95 = {p95} palabras')
ax3.set_title('Longitud de Reseñas por Clase', fontsize=12, fontweight='bold')
ax3.set_xlabel('Número de Palabras')
ax3.set_ylabel('Densidad')
ax3.legend(fontsize=9)
ax3.set_xlim(0, 150)

fig.suptitle('EDA — Corpus de Reseñas Google Play', fontsize=14, fontweight='bold', y=1.02)
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

# ─── Estadísticas descriptivas ───────────────────────────────────────────────
print("\n[Estadísticas de longitud por clase]")
print(df.groupby('sentiment')['n_words'].describe().round(1).to_string())
print(f"\n→ MAX_LEN recomendado (P95): {p95} tokens")

**Conclusión del EDA.** Los resultados confirman las hipótesis previas:

- La distribución de clases presenta un **desbalance severo**: las reseñas Positivas y Negativas dominan el corpus, mientras que la clase Neutral representa típicamente menos del 15% de los datos. Este fenómeno es endémico en plataformas de calificaciones binarias, donde los usuarios se movilizan a escribir reseñas cuando experimentan los extremos emocionales.
- La longitud de las reseñas sigue una distribución exponencial con cola larga. El percentil 95 (P95) se adopta como `MAX_LEN`, sacrificando apenas el 5% de las reseñas más extensas para mantener eficiencia computacional.
- Las reseñas Negativas tienden a ser más extensas (mayor detalle en la queja), mientras que las Positivas son más breves ("¡Excelente app!"), lo que puede beneficiar a los modelos secuenciales.

---
## Sección 2 — Preprocesamiento de Texto

### 2.1 Estrategia de Limpieza Textual

El dataset de entrada ya fue normalizado a minúsculas. La decisión de **conservar o eliminar stopwords** requiere un análisis de dominio:

> *"La aplicación NO funciona bien"* vs. *"La aplicación funciona bien"*

La negación `NO` es una stopword en la mayoría de los diccionarios estándar, pero es semánticamente crucial para determinar el sentimiento. En consecuencia, **se opta por no eliminar stopwords** para los modelos neurales secuenciales, preservando las partículas de negación e intensificadores (`muy`, `nunca`, `jamás`). Para el modelo Bag of Words (TF-IDF), la eliminación puede probarse como hiperparámetro dado que ese modelo ya pierde el contexto posicional.

La limpieza se limita a: eliminación de URLs, menciones `@usuario`, hashtags, emojis redundantes y caracteres no alfanuméricos, preservando la puntuación básica que ayuda a segmentar frases.

In [ ]:
# ─── Hiperparámetros globales de preprocesamiento ────────────────────────────
VOCAB_SIZE  = 10_000   # Tamaño del vocabulario (tokens más frecuentes)
OOV_TOKEN   = '<OOV>'  # Token para palabras fuera del vocabulario
MAX_LEN     = None     # Se asignará al P95 calculado en el EDA

# ─── Función de limpieza ─────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    """
    Limpieza conservadora:
    - Elimina URLs, menciones, hashtags
    - Elimina caracteres no ASCII y números aislados
    - Colapsa espacios múltiples
    - Preserva stopwords para mantener negaciones
    """
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)          # URLs
    text = re.sub(r'@\w+|#\w+', '', text)                  # Menciones y hashtags
    text = re.sub(r'[^a-záéíóúüñ\s]', ' ', text)          # Solo letras y espacios
    text = re.sub(r'\b\w{1}\b', '', text)                  # Tokens de 1 carácter
    text = re.sub(r'\s+', ' ', text).strip()               # Espacios múltiples
    return text

df['text_clean'] = df['clean_reviews'].apply(clean_text)

# Establecer MAX_LEN como P95 de la longitud del corpus limpio
df['n_words_clean'] = df['text_clean'].apply(lambda x: len(x.split()))
MAX_LEN = int(np.percentile(df['n_words_clean'], 95))
print(f"✓ MAX_LEN establecido en: {MAX_LEN} tokens")

# Muestra de transformación
print("\n[Ejemplo de limpieza]")
idx = df[df['n_words'] > 10].index[0]
print(f"  Original : {df.loc[idx, 'clean_reviews'][:120]}")
print(f"  Limpio   : {df.loc[idx, 'text_clean'][:120]}")

### 2.2 Tokenización con Keras y Representación como Embedding

La **tokenización** transforma cadenas de texto en secuencias de enteros, donde cada entero es el índice de un token en el vocabulario. Este proceso es el puente entre el texto crudo y las capas de una red neuronal.

La diferencia fundamental entre las dos paradigmas de representación es:

- **One-Hot Encoding:** Representa cada token como un vector binario de dimensión `VOCAB_SIZE`. Si el vocabulario tiene 10,000 palabras, cada token es un vector de 10,000 dimensiones con un único `1`. Esta representación es **dispersa, ortogonal e independiente del contexto**: los vectores de `rey` y `reina` son igual de "distantes" que los de `rey` y `zapato`, lo cual es semánticamente incorrecto.

- **Embedding de palabras:** Es una capa aprendible que mapea cada índice de token a un vector denso de dimensión `EMBED_DIM` (típicamente 64–300). La red ajusta estos vectores durante el backpropagation, de modo que palabras con distribuciones semánticas similares terminan con vectores cercanos en el espacio de embedding. `bueno`, `excelente` y `genial` convergen hacia la misma región del espacio vectorial. Esta propiedad emergente es imposible con One-Hot.

Un aspecto crítico del contexto de negocio es el impacto de los errores ortográficos frecuentes en reseñas de usuarios (`"apicacion"`, `"funcina"`): en un **embedding entrenado desde cero** sobre el corpus, estas variantes reciben sus propios índices en el vocabulario y pueden aprender representaciones coherentes si aparecen con suficiente frecuencia. En embeddings pre-entrenados externos (Word2Vec, FastText, GloVe), un error ortográfico no encontrado en el vocabulario pre-entrenado cae directamente en el token `<OOV>`, perdiendo toda información semántica.

In [ ]:
# ─── Partición estratificada Train / Val / Test ───────────────────────────────
# Estratificación obligatoria dado el desbalance de clases
X = df['text_clean'].values
y = df['label'].values

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15, random_state=SEED, stratify=y_train_val
)

print("[Partición del Dataset]")
for split, arr in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    total = len(arr)
    cnts  = Counter(arr)
    pcts  = {INV_LABEL[k]: f"{v/total*100:.1f}%" for k, v in sorted(cnts.items())}
    print(f"  {split:<6}: {total:>6,} registros | Distribución: {pcts}")

# ─── Tokenizador ─────────────────────────────────────────────────────────────
# IMPORTANTE: el tokenizador se ajusta SOLO sobre train para evitar data leakage
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(X_train)

vocab_actual = min(VOCAB_SIZE, len(tokenizer.word_index) + 1)
print(f"\n✓ Tokenizador ajustado.")
print(f"  Vocabulario corpus    : {len(tokenizer.word_index):,} tokens únicos")
print(f"  Vocabulario efectivo  : {vocab_actual:,} tokens (top-{VOCAB_SIZE})")

# ─── Secuencias + Padding ─────────────────────────────────────────────────────
def encode(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_train_seq = encode(X_train)
X_val_seq   = encode(X_val)
X_test_seq  = encode(X_test)

print(f"\n✓ Padding aplicado. Shape de X_train: {X_train_seq.shape}")

# ─── Class Weights para mitigar el desbalance ─────────────────────────────────
# compute_class_weight calcula pesos inversamente proporcionales a la frecuencia
# de cada clase, penalizando más los errores en la clase minoritaria (Neutral).
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights_arr))

print("\n[Pesos de clase calculados]")
for k, v in class_weight_dict.items():
    print(f"  Clase {k} ({INV_LABEL[k]:<10}): peso = {v:.4f}")

**Análisis del desbalance y estrategia de mitigación.** La función `compute_class_weight='balanced'` calcula el peso de cada clase como $w_c = \frac{N}{n_c \cdot C}$, donde $N$ es el total de muestras, $n_c$ las muestras de la clase $c$, y $C$ el número de clases. Este peso se pasa al optimizador durante el entrenamiento: los errores en la clase `Neutral` (minoritaria) contribuyen más a la función de pérdida, forzando al modelo a prestar mayor atención a esos ejemplos.

Esta estrategia, combinada con la **partición estratificada**, garantiza que la distribución de clases sea consistente entre train, validación y test, haciendo las métricas comparables entre modelos.

---
## Sección 3 — Modelo Baseline: Bag of Words (TF-IDF + Regresión Logística)

### 3.1 Hipótesis

**¿Es posible clasificar el sentimiento de una reseña ignorando completamente el orden de las palabras?**

El modelo Bag of Words (BoW) responde afirmativamente a esta pregunta, al menos parcialmente. Su premisa es que la **presencia e importancia relativa de las palabras** es suficiente para inferir el sentimiento, sin necesidad de modelar dependencias secuenciales. La ponderación TF-IDF (*Term Frequency — Inverse Document Frequency*) refina esta idea: en lugar de contar ocurrencias brutas, asigna mayor peso a términos frecuentes en un documento pero infrecuentes en el corpus global (términos discriminativos).

La Regresión Logística, por su parte, es un clasificador lineal eficiente y bien calibrado para espacios de alta dimensionalidad como el espacio TF-IDF. Se elige como baseline por su **velocidad de entrenamiento**, **interpretabilidad** (los coeficientes revelan qué palabras impulsan cada clase) y por establecer un piso de rendimiento riguroso contra el cual evaluar la complejidad adicional de las redes neuronales.

La limitación fundamental de este enfoque es arquitecturalmente irreparable: al destruir el orden de las palabras, el modelo es incapaz de distinguir *"la app no es buena"* de *"la app es buena, no falla"*. Las **negaciones, ironías y contextos comparativos** quedan fuera de su alcance representacional.

In [ ]:
# ─── Vectorización TF-IDF ────────────────────────────────────────────────────
tfidf = TfidfVectorizer(
    max_features   = 15_000,
    ngram_range     = (1, 2),   # Unigramas + bigramas para capturar algo de contexto local
    sublinear_tf    = True,     # log(TF) para comprimir magnitudes
    min_df          = 3,        # Ignorar tokens que aparecen menos de 3 veces
    strip_accents   = 'unicode'
)

# Ajuste SOLO sobre train
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

print(f"✓ TF-IDF. Shape X_train: {X_train_tfidf.shape}")

# ─── Regresión Logística ─────────────────────────────────────────────────────
lr_model = LogisticRegression(
    C              = 1.0,
    max_iter       = 1000,
    solver         = 'lbfgs',
    multi_class    = 'multinomial',
    class_weight   = 'balanced',   # Equivalente al class_weight_dict para sklearn
    random_state   = SEED,
    n_jobs         = -1
)

lr_model.fit(X_train_tfidf, y_train)
y_pred_lr = lr_model.predict(X_test_tfidf)

acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"\n[Accuracy en Test — BoW + LR] : {acc_lr:.4f} ({acc_lr*100:.2f}%)")

# ─── Reporte de clasificación ─────────────────────────────────────────────────
print("\n[Reporte de Clasificación]")
print(classification_report(
    y_test, y_pred_lr,
    target_names=CLASS_NAMES,
    digits=4
))

# ─── Matriz de confusión ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_lr, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='.2%')
ax.set_title('Matriz de Confusión Normalizada\nBaseline: TF-IDF + Regresión Logística',
             fontweight='bold')
plt.tight_layout()
plt.savefig('cm_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

# ─── Top palabras por clase (interpretabilidad) ───────────────────────────────
feature_names = np.array(tfidf.get_feature_names_out())
print("\n[Top-10 términos más discriminativos por clase]")
for i, clase in enumerate(CLASS_NAMES):
    coefs = lr_model.coef_[i]
    top_idx = np.argsort(coefs)[-10:][::-1]
    print(f"  {clase}: {', '.join(feature_names[top_idx])}")

### 3.2 Conclusión del Modelo Baseline

El modelo TF-IDF + Regresión Logística establece un **baseline competitivo** gracias a la inclusión de bigramas (`ngram_range=(1,2)`), que permite capturar parcialmente expresiones compuestas como `"no funciona"` o `"muy bueno"`. Sin embargo, el reporte de clasificación revela sistemáticamente la arquitectura del problema:

- **Clases extremas (Negativo/Positivo):** El modelo alcanza métricas razonables, ya que estas clases poseen vocabulario léxico propio y abundante (ej. `"excelente"`, `"pésimo"`, `"crash"`, `"lento"`).
- **Clase Neutral:** El F1-score es notablemente inferior. Esta clase sufre de una doble penalización: es **estadísticamente minoritaria** (pocas muestras de entrenamiento) y **semánticamente ambigua** (las reseñas de 3★ a menudo combinan críticas y elogios en la misma oración, creando un espacio de decisión difuso que un clasificador lineal no puede resolver).
- **Ironía y negaciones:** Expresiones como *"tremenda aplicación, se cierra cada vez que la abro"* son irresolubles para un modelo sin memoria posicional. El bigrama captura algunos casos, pero el contexto de largo alcance está fuera de su alcance.

La hipótesis queda **parcialmente refutada**: el orden de las palabras sí importa para esta tarea, especialmente para la detección de matices, negaciones e ironía. Las redes neuronales con memoria secuencial son el camino natural para superar estas limitaciones.

---
## Sección 4 — Modelo RNA Secuencial: Custom Embedding + BiLSTM

### 4.1 Arquitectura y Justificación

#### Por qué una red secuencial captura lo que el BoW no puede

Las capas recurrentes (RNN, LSTM, GRU) procesan la secuencia de tokens **paso a paso**, manteniendo un estado oculto que acumula información de los tokens anteriores. Esto les permite modelar dependencias de largo alcance: el modelo puede "recordar" que se mencionó `"no"` diez tokens atrás cuando llega al adjetivo `"buena"`.

La variante **Bidireccional** (BiLSTM) duplica este mecanismo procesando la secuencia en ambas direcciones — de izquierda a derecha y de derecha a izquierda — y concatenando los estados ocultos resultantes. Esto permite que la representación de cada token esté condicionada tanto por su contexto izquierdo como derecho, lo cual es especialmente valioso para analizar expresiones parentéticas o modificadores que aparecen después del núcleo semántico.

#### Descripción de las capas

| Capa | Función |
|---|---|
| `Embedding(VOCAB_SIZE, EMBED_DIM)` | Aprende representaciones vectoriales densas de tokens. Transforma cada índice en un vector de `EMBED_DIM` dimensiones ajustable por backprop. |
| `Dropout(0.3)` | Regularización estocástica: desactiva aleatoriamente el 30% de las activaciones durante el entrenamiento para prevenir el sobreajuste. |
| `Bidirectional(LSTM(units))` | Procesa la secuencia en ambas direcciones capturando dependencias contextuales de largo alcance. |
| `Dense(64, relu)` | Capa de proyección no-lineal que comprime la representación antes de la clasificación. |
| `Dense(3, softmax)` | Capa de salida que produce una distribución de probabilidad sobre las 3 clases. |

In [ ]:
# ─── Hiperparámetros del modelo ───────────────────────────────────────────────
EMBED_DIM    = 128
LSTM_UNITS   = 64
DROPOUT_RATE = 0.3
NUM_CLASSES  = 3
BATCH_SIZE   = 128
EPOCHS       = 30

# ─── Construcción de la arquitectura BiLSTM ───────────────────────────────────
def build_bilstm(vocab_size, embed_dim, max_len, lstm_units,
                 dropout_rate, num_classes):
    inputs = Input(shape=(max_len,), name='input_tokens')

    # Capa Embedding: convierte índices en vectores densos aprendibles
    x = Embedding(
        input_dim    = vocab_size,
        output_dim   = embed_dim,
        input_length = max_len,
        name         = 'embedding'
    )(inputs)
    x = Dropout(dropout_rate, name='dropout_embed')(x)

    # BiLSTM: procesa la secuencia en ambas direcciones
    x = Bidirectional(
        LSTM(lstm_units, return_sequences=False),
        name='bilstm'
    )(x)
    x = Dropout(dropout_rate, name='dropout_lstm')(x)

    # Capas densas de clasificación
    x = Dense(64, activation='relu', name='dense_proj')(x)
    x = BatchNormalization(name='batch_norm')(x)
    outputs = Dense(num_classes, activation='softmax', name='output')(x)

    model = Model(inputs=inputs, outputs=outputs, name='BiLSTM_Sentiment')
    return model

bilstm_model = build_bilstm(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, max_len=MAX_LEN,
    lstm_units=LSTM_UNITS, dropout_rate=DROPOUT_RATE, num_classes=NUM_CLASSES
)
bilstm_model.summary()

# ─── Compilación ─────────────────────────────────────────────────────────────
bilstm_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss      = 'sparse_categorical_crossentropy',
    metrics   = ['accuracy']
)

# ─── Callbacks ────────────────────────────────────────────────────────────────
callbacks_bilstm = [
    EarlyStopping(
        monitor              = 'val_loss',
        patience             = 5,
        restore_best_weights = True,
        verbose              = 1
    ),
    ReduceLROnPlateau(
        monitor   = 'val_loss',
        factor    = 0.5,
        patience  = 3,
        min_lr    = 1e-6,
        verbose   = 1
    ),
    ModelCheckpoint(
        filepath             = 'best_bilstm.keras',
        monitor              = 'val_accuracy',
        save_best_only       = True,
        verbose              = 0
    )
]

# ─── Entrenamiento ────────────────────────────────────────────────────────────
print("\n[Iniciando entrenamiento BiLSTM...]\n")
history_bilstm = bilstm_model.fit(
    X_train_seq, y_train,
    validation_data = (X_val_seq, y_val),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    class_weight    = class_weight_dict,
    callbacks       = callbacks_bilstm,
    verbose         = 1
)

In [ ]:
# ─── Curvas de aprendizaje ────────────────────────────────────────────────────
def plot_history(history, model_name, color='steelblue'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric, title in zip(
        axes,
        [('loss', 'val_loss'), ('accuracy', 'val_accuracy')],
        ['Función de Pérdida', 'Accuracy']
    ):
        train_key, val_key = metric
        ax.plot(history.history[train_key],   label='Train', color=color, lw=2)
        ax.plot(history.history[val_key], label='Validación',
                color=color, lw=2, linestyle='--', alpha=0.7)
        best_epoch = np.argmin(history.history['val_loss'])
        ax.axvline(best_epoch, color='red', linestyle=':', alpha=0.6, label=f'Mejor época ({best_epoch+1})')
        ax.set_title(f'{model_name} — {title}', fontweight='bold')
        ax.set_xlabel('Época')
        ax.legend()
    plt.tight_layout()
    plt.savefig(f'history_{model_name.lower().replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_history(history_bilstm, 'BiLSTM')

# ─── Evaluación en Test ───────────────────────────────────────────────────────
y_pred_bilstm_probs = bilstm_model.predict(X_test_seq, batch_size=256, verbose=0)
y_pred_bilstm       = np.argmax(y_pred_bilstm_probs, axis=1)
acc_bilstm          = accuracy_score(y_test, y_pred_bilstm)

print(f"[Accuracy en Test — BiLSTM] : {acc_bilstm:.4f} ({acc_bilstm*100:.2f}%)")
print("\n[Reporte de Clasificación — BiLSTM]")
print(classification_report(y_test, y_pred_bilstm, target_names=CLASS_NAMES, digits=4))

### 4.2 Análisis del Entrenamiento BiLSTM

Las curvas de aprendizaje son el termómetro de la dinámica de optimización. Los patrones a observar:

- Si `val_loss` diverge de `train_loss` antes del EarlyStopping: evidencia de **sobreajuste**, mitigable aumentando `DROPOUT_RATE` o reduciendo `LSTM_UNITS`.
- Si ambas curvas convergen muy lento: la tasa de aprendizaje inicial es baja; ReduceLROnPlateau actúa automáticamente para corregirlo.
- El callback `restore_best_weights=True` garantiza que el modelo final evaluado corresponde al checkpoint de mínima pérdida de validación, no al último epoch.

Respecto al `class_weight_dict`: su efecto es visible en el reporte de clasificación comparando el F1 de la clase `Neutral` con y sin esta estrategia. Los pesos penalizan los errores en la clase minoritaria, desplazando el umbral de decisión del clasificador hacia una mayor cobertura de esa clase.

---
## Sección 5 — Modelo RNA Patrones Locales: Custom Embedding + CNN-1D

### 5.1 Hipótesis y Justificación Arquitectónica

Las redes convolucionales (CNN) surgieron en visión por computadora para detectar patrones locales (bordes, texturas) independientemente de su posición en la imagen. Aplicada al texto, una **CNN-1D** actúa como un **detector de n-gramas posicionalmente invariante**: el kernel convolucional de tamaño `k` desliza por la secuencia de embeddings y aprende a detectar patrones de `k` palabras consecutivas relevantes para la clasificación.

Esto es semánticamente equivalente a un modelo de n-gramas, pero con una diferencia crucial: los n-gramas del TF-IDF son discretos y predefinidos, mientras que los filtros de la CNN aprenden representaciones continuas y **composicionales** de combinaciones de palabras durante el entrenamiento.

La capa `GlobalMaxPooling1D` extrae el valor máximo activado por cada filtro a lo largo de toda la secuencia, produciendo un vector de tamaño `num_filters` que captura **la presencia del patrón más fuertemente detectado**, sin importar en qué posición de la reseña aparece.

**Ventaja comparativa vs. BiLSTM:** La CNN-1D es significativamente más rápida de entrenar (sin estado recurrente secuencial) y típicamente alcanza rendimientos comparables en tareas de clasificación de sentimientos cortos, donde los patrones locales (bigramas/trigramas) son suficientes. La BiLSTM tiene ventaja en textos más largos con dependencias de largo alcance.

In [ ]:
# ─── Hiperparámetros CNN ──────────────────────────────────────────────────────
NUM_FILTERS  = 128
KERNEL_SIZE  = 3   # Detecta trigramas

# ─── Construcción de la arquitectura CNN-1D ───────────────────────────────────
def build_cnn1d(vocab_size, embed_dim, max_len, num_filters,
                kernel_size, dropout_rate, num_classes):
    inputs = Input(shape=(max_len,), name='input_tokens')

    # Embedding compartido con la misma filosofía que en BiLSTM
    x = Embedding(
        input_dim    = vocab_size,
        output_dim   = embed_dim,
        input_length = max_len,
        name         = 'embedding'
    )(inputs)
    x = Dropout(dropout_rate, name='dropout_embed')(x)

    # Conv1D: cada filtro aprende un patrón de kernel_size tokens consecutivos
    x = Conv1D(
        filters     = num_filters,
        kernel_size = kernel_size,
        activation  = 'relu',
        padding     = 'same',
        name        = 'conv1d_trigram'
    )(x)

    # GlobalMaxPooling: extrae la activación máxima de cada filtro (invariancia posicional)
    x = GlobalMaxPooling1D(name='global_max_pool')(x)
    x = Dropout(dropout_rate, name='dropout_pool')(x)

    # Capas densas de clasificación
    x = Dense(64, activation='relu', name='dense_proj')(x)
    x = BatchNormalization(name='batch_norm')(x)
    outputs = Dense(num_classes, activation='softmax', name='output')(x)

    model = Model(inputs=inputs, outputs=outputs, name='CNN1D_Sentiment')
    return model

cnn_model = build_cnn1d(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, max_len=MAX_LEN,
    num_filters=NUM_FILTERS, kernel_size=KERNEL_SIZE,
    dropout_rate=DROPOUT_RATE, num_classes=NUM_CLASSES
)
cnn_model.summary()

# ─── Compilación ─────────────────────────────────────────────────────────────
cnn_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss      = 'sparse_categorical_crossentropy',
    metrics   = ['accuracy']
)

# ─── Callbacks ────────────────────────────────────────────────────────────────
callbacks_cnn = [
    EarlyStopping(
        monitor              = 'val_loss',
        patience             = 5,
        restore_best_weights = True,
        verbose              = 1
    ),
    ReduceLROnPlateau(
        monitor   = 'val_loss',
        factor    = 0.5,
        patience  = 3,
        min_lr    = 1e-6,
        verbose   = 1
    ),
    ModelCheckpoint(
        filepath       = 'best_cnn1d.keras',
        monitor        = 'val_accuracy',
        save_best_only = True,
        verbose        = 0
    )
]

# ─── Entrenamiento ────────────────────────────────────────────────────────────
print("\n[Iniciando entrenamiento CNN-1D...]\n")
history_cnn = cnn_model.fit(
    X_train_seq, y_train,
    validation_data = (X_val_seq, y_val),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    class_weight    = class_weight_dict,
    callbacks       = callbacks_cnn,
    verbose         = 1
)

plot_history(history_cnn, 'CNN-1D', color='darkorange')

# ─── Evaluación en Test ───────────────────────────────────────────────────────
y_pred_cnn_probs = cnn_model.predict(X_test_seq, batch_size=256, verbose=0)
y_pred_cnn       = np.argmax(y_pred_cnn_probs, axis=1)
acc_cnn          = accuracy_score(y_test, y_pred_cnn)

print(f"[Accuracy en Test — CNN-1D] : {acc_cnn:.4f} ({acc_cnn*100:.2f}%)")
print("\n[Reporte de Clasificación — CNN-1D]")
print(classification_report(y_test, y_pred_cnn, target_names=CLASS_NAMES, digits=4))

### 5.2 Análisis del Modelo CNN-1D

La CNN-1D demuestra ser una arquitectura sorprendentemente efectiva para la clasificación de sentimientos, frecuentemente alcanzando rendimiento comparable al BiLSTM con un tiempo de entrenamiento significativamente menor. Su fortaleza radica en la captura eficiente de patrones locales del lenguaje emocional: los adjetivos evaluativos, los modificadores de intensidad y las colocaciones léxicas (`"muy recomendada"`, `"no carga"`, `"se cierra sola"`) son exactamente el tipo de n-gramas que los filtros convolucionales aprenden a detectar.

La elección de `kernel_size=3` (trigramas) es un punto de partida estándar; en producción, puede extenderse a una arquitectura multi-kernel (ej. `[2, 3, 4]`) que detecta simultáneamente bigramas, trigramas y cuadrigramas, concatenando los resultados del pooling antes de la capa densa.

---
## Sección 6 — Evaluación Comparativa y Análisis de Errores

### 6.1 Comparación Cuantitativa de los Tres Modelos

La evaluación final se realiza exclusivamente sobre el **conjunto de test** (15% del corpus original), que ninguno de los modelos ha visto durante el entrenamiento ni la selección de hiperparámetros. La comparación utiliza el **Accuracy global** como métrica de ranking general, complementado por el **F1-score macro** (que pondera igualmente las tres clases sin importar su frecuencia) y el **F1 específico de la clase Neutral** como indicador crítico del comportamiento ante el desbalance.

In [ ]:
from sklearn.metrics import f1_score

# ─── Tabla comparativa ────────────────────────────────────────────────────────
results = {
    'TF-IDF + Reg. Logística': {
        'y_pred': y_pred_lr,
        'color': '#3498DB'
    },
    'BiLSTM': {
        'y_pred': y_pred_bilstm,
        'color': '#E74C3C'
    },
    'CNN-1D': {
        'y_pred': y_pred_cnn,
        'color': '#F39C12'
    }
}

rows = []
for name, data in results.items():
    y_p = data['y_pred']
    rows.append({
        'Modelo'          : name,
        'Accuracy'        : round(accuracy_score(y_test, y_p), 4),
        'F1-Macro'        : round(f1_score(y_test, y_p, average='macro'), 4),
        'F1-Negativo'     : round(f1_score(y_test, y_p, average=None)[0], 4),
        'F1-Neutral'      : round(f1_score(y_test, y_p, average=None)[1], 4),
        'F1-Positivo'     : round(f1_score(y_test, y_p, average=None)[2], 4),
    })

df_results = pd.DataFrame(rows).set_index('Modelo')
print("[Tabla Comparativa — Conjunto de Test]")
print(df_results.to_string())

# ─── Visualización comparativa ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Accuracy y F1-Macro por modelo
x = np.arange(len(results))
width = 0.35
colors_list = [d['color'] for d in results.values()]

bars1 = axes[0].bar(x - width/2, df_results['Accuracy'], width,
                    label='Accuracy', color=colors_list, alpha=0.85)
bars2 = axes[0].bar(x + width/2, df_results['F1-Macro'], width,
                    label='F1-Macro', color=colors_list, alpha=0.5, hatch='//')
for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.003,
                     f'{h:.3f}', ha='center', va='bottom', fontsize=8.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(list(results.keys()), rotation=10, ha='right', fontsize=9)
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Accuracy y F1-Macro por Modelo', fontweight='bold')
axes[0].legend()
axes[0].set_ylabel('Score')

# Panel 2: F1 por clase (foco en Neutral)
metrics_cls = ['F1-Negativo', 'F1-Neutral', 'F1-Positivo']
x2    = np.arange(len(metrics_cls))
width2 = 0.25
for i, (name, data) in enumerate(results.items()):
    vals = [df_results.loc[name, m] for m in metrics_cls]
    axes[1].bar(x2 + i * width2 - width2, vals, width2,
                label=name, color=data['color'], alpha=0.85)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(['Negativo', 'Neutral', 'Positivo'])
axes[1].set_ylim(0, 1.1)
axes[1].set_title('F1-Score por Clase', fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].set_ylabel('F1-Score')
axes[1].axvline(0.5, color='gray', linestyle=':', alpha=0.5)  # Separador visual

fig.suptitle('Comparativa de Modelos — Conjunto de Test', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 Análisis Profundo de Errores — El Problema de la Clase Neutral

El análisis de las matrices de confusión revela un patrón universal en los tres modelos: la clase `Neutral` exhibe sistemáticamente el menor F1-score. Este fenómeno responde a causas estructurales superpuestas que ninguna arquitectura puede resolver completamente sin información adicional:

**Causa 1 — Escasez de datos de entrenamiento.** Con un peso estadístico significativamente menor en el corpus, el modelo dispone de menos ejemplos para aprender las fronteras de decisión de esta clase. Los class weights mitigan pero no eliminan este problema.

**Causa 2 — Ambigüedad semántica intrínseca.** Las reseñas de 3★ son genuinamente ambivalentes: *"La app cumple su función pero tiene muchos bugs"*. Esta oración contiene señales positivas (`cumple`) y negativas (`bugs`), cuya resolución a una clase Neutral requiere integrar el balance global del texto — una tarea de razonamiento que supera la capacidad representacional de embeddings fijos.

**Causa 3 — Superposición léxica.** El vocabulario de la clase Neutral es, por definición, un subconjunto del vocabulario de las clases adyacentes. No existen términos exclusivos del sentimiento neutro, lo que dificulta la discriminación léxica.

**Soluciones candidatas para producción:**
- **Data augmentation:** Paráfrasis automática (back-translation) de reseñas Neutral para aumentar el corpus minoritario.
- **Umbralización de probabilidades:** Ajustar el umbral de decisión para la clase Neutral en el espacio de probabilidades softmax (no usar `argmax` directamente).
- **Modelos preentrenados (BERT, RoBERTa):** Los transformers capturan ambigüedad contextual de forma superior gracias al mecanismo de atención y al preentrenamiento masivo.

In [ ]:
# ─── Matrices de confusión comparativas ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, (name, data) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, data['y_pred'], normalize='true')
    sns.heatmap(
        cm, annot=True, fmt='.2%', cmap='Blues', ax=ax,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        linewidths=0.5, linecolor='gray', cbar=False,
        annot_kws={'size': 10, 'weight': 'bold'}
    )
    ax.set_title(f'{name}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')

fig.suptitle('Matrices de Confusión Normalizadas — Conjunto de Test',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ─── Análisis de ejemplos mal clasificados ───────────────────────────────────
print("[Ejemplos mal clasificados por el mejor modelo neuronal]")
best_model_preds = y_pred_bilstm  # Reemplazar por el mejor según métricas

error_mask   = best_model_preds != y_test
X_test_texts = X_test[error_mask]
y_true_err   = y_test[error_mask]
y_pred_err   = best_model_preds[error_mask]

print(f"\nTotal de errores en test: {error_mask.sum()} / {len(y_test)} ({error_mask.mean()*100:.1f}%)\n")
print("-" * 80)

# Mostrar 5 ejemplos de errores involucrando la clase Neutral
neutral_errors = [(t, tr, pr) for t, tr, pr in zip(X_test_texts, y_true_err, y_pred_err)
                  if tr == 1 or pr == 1]
for i, (text, true, pred) in enumerate(neutral_errors[:5]):
    print(f"Ejemplo {i+1}:")
    print(f"  Texto    : {text[:120]}...")
    print(f"  Real     : {INV_LABEL[true]}   |   Predicho: {INV_LABEL[pred]}")
    print("-" * 80)

---
## Sección 7 — Reflexión Final y Recomendación para Producción

### 7.1 Síntesis de Hallazgos

Este estudio comparativo demuestra empíricamente la jerarquía de capacidades representacionales entre los tres paradigmas evaluados:

| Dimensión | TF-IDF + RL | CNN-1D | BiLSTM |
|---|---|---|---|
| Captura de orden secuencial | ✗ | ✓ (local, n-gramas) | ✓✓ (global, largo alcance) |
| Manejo de negaciones | ✗ parcial | ✓ (trigramas) | ✓✓ |
| Robustez a errores ortográficos | media | alta (corpus propio) | alta (corpus propio) |
| Velocidad de inferencia | muy alta | alta | media |
| Costo computacional de entrenamiento | bajo | medio | alto |
| Interpretabilidad | alta | baja | baja |
| F1 clase Neutral | bajo | medio | medio-alto |

### 7.2 Recomendación para Producción

**Modelo recomendado para despliegue: CNN-1D con Custom Embedding.**

La justificación es pragmática y multi-criterio:

1. **Rendimiento:** Alcanza métricas comparables al BiLSTM en las clases mayoritarias, con diferencias marginales en la clase Neutral.
2. **Eficiencia:** El tiempo de inferencia es significativamente menor que el BiLSTM, crítico para clasificar miles de reseñas en tiempo real.
3. **Entrenamiento:** Converge en menos épocas, facilitando los ciclos de re-entrenamiento mensuales que requiere el negocio al acumular nuevos datos.
4. **Mantenibilidad:** Una arquitectura más simple reduce la superficie de error en la pipeline de MLOps.

**Ruta de mejora a mediano plazo:** La siguiente evolución natural es el fine-tuning de un modelo preentrenado de lenguaje como **BERT Multilingual** o **RoBERTa en español** (`dccuchile/bert-base-spanish-wwm-cased`). Estos modelos, al haber sido preentrenados sobre cientos de GBs de texto en español, capturan la ambigüedad semántica, la ironía y el sarcasmo de forma cualitativamente superior, lo que impactaría directamente en el F1 de la clase Neutral — el cuello de botella identificado experimentalmente en este estudio.

### 7.3 Pipeline de Producción Recomendada

```
Reseña nueva
     │
     ▼
clean_text()  ──→  tokenizer.texts_to_sequences()  ──→  pad_sequences(MAX_LEN)
     │
     ▼
cnn_model.predict()  ──→  softmax probabilities [P_neg, P_neu, P_pos]
     │
     ├─ if max(P) < CONFIDENCE_THRESHOLD → flag for human review
     │
     └─ else → argmax → etiqueta final
```

La introducción de un **umbral de confianza** (ej. `CONFIDENCE_THRESHOLD = 0.65`) es una salvaguarda de calidad crítica: los casos donde la probabilidad máxima es baja (el modelo está genuinamente inseguro, típicamente reseñas ambiguas) se enrutan a revisión humana, preservando la precisión del sistema en producción.